# 03 Keyword Analysis

This notebook analyzes the keyword-search results and answers the project's
expected outcomes.

## Expected Outcomes

- Identify customers’ concerns about loan and insurance products.
- Determine why customers contacted the chatbot.
- Identify questions and interest related to marketing campaigns.
- Extract common product conditions, requirements, and customer needs.
- Provide insights that can help the marketing team improve campaigns and
  better address customer demand.

The analysis is performed at both the message level and the `userID` level.
Message-level analysis measures how frequently topics appear, while
`userID`-level analysis reduces repeated counting when the same customer sends
multiple messages.

Import Libraries

In [1]:
import pandas as pd
from pathlib import Path

Load the keyword results

In [2]:
project_root = Path.cwd().parent.parent

keyword_results_path = (
    project_root
    / "data"
    / "processed"
    / "keyword_search_results.csv"
)

print("File exists:", keyword_results_path.exists())

File exists: True


Load the file:

In [3]:
keyword_df = pd.read_csv(keyword_results_path)

print("Keyword results loaded successfully.")

Keyword results loaded successfully.


Restore Boolean columns

In [4]:
category_columns = [
    "loan_product",
    "insurance_product",
    "eligibility",
    "application_process",
    "required_documents",
    "approval_and_status",
    "interest_and_fees",
    "credit_limit",
    "payment_and_installment",
    "insurance_premium",
    "insurance_claim",
    "coverage_and_conditions",
    "renewal_and_cancellation",
    "campaign_or_promotion",
    "campaign_interest",
    "branch_or_contact"
]

In [5]:
for column in category_columns:
    if keyword_df[column].dtype == "object":
        keyword_df[column] = (
            keyword_df[column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
            .fillna(False)
            .astype(bool)
        )

Create conversation-level results

In [6]:
conversation_results = (
    keyword_df.groupby("userID")[category_columns]
    .max()
    .reset_index()
)

print("Conversation-level results created successfully.")

Conversation-level results created successfully.


readable category names

In [7]:
category_labels = {
    "loan_product": "Loan product",
    "insurance_product": "Insurance product",
    "eligibility": "Eligibility",
    "application_process": "Application process",
    "required_documents": "Required documents",
    "approval_and_status": "Approval and status",
    "interest_and_fees": "Interest and fees",
    "credit_limit": "Credit limit",
    "payment_and_installment": "Payment and installment",
    "insurance_premium": "Insurance premium",
    "insurance_claim": "Insurance claim",
    "coverage_and_conditions": "Coverage and conditions",
    "renewal_and_cancellation": "Renewal and cancellation",
    "campaign_or_promotion": "Campaign or promotion",
    "campaign_interest": "Campaign interest",
    "branch_or_contact": "Branch or contact"
}

Outcome 1: Customer Concerns About Loan and Insurance Products

This section identifies the most common concern categories appearing in
conversations associated with loan and insurance products.

In [8]:
concern_columns = [
    "eligibility",
    "application_process",
    "required_documents",
    "approval_and_status",
    "interest_and_fees",
    "credit_limit",
    "payment_and_installment",
    "insurance_premium",
    "insurance_claim",
    "coverage_and_conditions",
    "renewal_and_cancellation"
]

Loan Concerns

In [9]:
loan_concerns = (
    conversation_results.loc[
        conversation_results["loan_product"],
        concern_columns
    ]
    .sum()
    .sort_values(ascending=False)
    .rename("conversation_count")
    .reset_index()
    .rename(columns={"index": "concern"})
)

loan_concerns["concern"] = loan_concerns["concern"].map(
    category_labels
)

loan_concerns

,concern,conversation_count
0,Coverage and conditions,159
1,Application process,125
2,Required documents,115
3,Payment and installment,106
4,Credit limit,36
5,Interest and fees,35
6,Eligibility,27
7,Renewal and cancellation,17
8,Approval and status,8
9,Insurance premium,6


In [10]:
insurance_concerns = (
    conversation_results.loc[
        conversation_results["insurance_product"],
        concern_columns
    ]
    .sum()
    .sort_values(ascending=False)
    .rename("conversation_count")
    .reset_index()
    .rename(columns={"index": "concern"})
)

insurance_concerns["concern"] = insurance_concerns["concern"].map(
    category_labels
)

insurance_concerns

,concern,conversation_count
0,Payment and installment,24
1,Insurance premium,23
2,Coverage and conditions,18
3,Application process,8
4,Required documents,7
5,Interest and fees,2
6,Eligibility,1
7,Credit limit,1
8,Approval and status,1
9,Insurance claim,1


direct answer:

In [11]:
top_loan_concerns = loan_concerns.head(3)["concern"].tolist()
top_insurance_concerns = insurance_concerns.head(3)["concern"].tolist()

print("Top detected loan concerns:")
print(", ".join(top_loan_concerns))

print("\nTop detected insurance concerns:")
print(", ".join(top_insurance_concerns))

Top detected loan concerns:
Coverage and conditions, Application process, Required documents

Top detected insurance concerns:
Payment and installment, Insurance premium, Coverage and conditions


Expected outcome 2: Why customers contacted the chatbot

In [12]:
contact_reason_columns = [
    "eligibility",
    "application_process",
    "required_documents",
    "approval_and_status",
    "interest_and_fees",
    "credit_limit",
    "payment_and_installment",
    "insurance_premium",
    "insurance_claim",
    "coverage_and_conditions",
    "renewal_and_cancellation",
    "campaign_or_promotion",
    "campaign_interest",
    "branch_or_contact"
]

contact_reasons = (
    conversation_results[contact_reason_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("conversation_count")
    .reset_index()
    .rename(columns={"index": "contact_reason"})
)

contact_reasons["contact_reason"] = contact_reasons[
    "contact_reason"
].map(category_labels)

contact_reasons

,contact_reason,conversation_count
0,Coverage and conditions,166
1,Branch or contact,159
2,Payment and installment,145
3,Campaign interest,140
4,Application process,139
5,Required documents,122
6,Interest and fees,37
7,Credit limit,36
8,Campaign or promotion,33
9,Eligibility,31


Direct answer:

In [13]:
top_contact_reasons = contact_reasons.head(5)[
    "contact_reason"
].tolist()

print("The leading detected reasons for contacting the chatbot were:")
print(", ".join(top_contact_reasons))

The leading detected reasons for contacting the chatbot were:
Coverage and conditions, Branch or contact, Payment and installment, Campaign interest, Application process


Expected outcome 3: Campaign questions and interest

In [14]:
campaign_results = pd.DataFrame({
    "campaign_category": [
        "Campaign or promotion questions",
        "Campaign interest"
    ],
    "conversation_count": [
        conversation_results["campaign_or_promotion"].sum(),
        conversation_results["campaign_interest"].sum()
    ]
})

campaign_results

,campaign_category,conversation_count
0,Campaign or promotion questions,33
1,Campaign interest,140


Compare the two campaign indicators:

In [15]:
campaign_comparison = pd.crosstab(
    conversation_results["campaign_or_promotion"],
    conversation_results["campaign_interest"],
    rownames=["Campaign mentioned"],
    colnames=["Campaign interest detected"]
)

campaign_comparison

Campaign interest detected,False,True
Campaign mentioned,,
False,740,133
True,26,7


Direct answer:

In [16]:
campaign_questions = conversation_results[
    "campaign_or_promotion"
].sum()

campaign_interest = conversation_results[
    "campaign_interest"
].sum()

print(
    "Campaign-related questions were detected in",
    campaign_questions,
    "conversations."
)

print(
    "Campaign-interest language was detected in",
    campaign_interest,
    "conversations."
)

Campaign-related questions were detected in 33 conversations.
Campaign-interest language was detected in 140 conversations.


Expected outcome 4: Conditions, requirements, and needs

In [17]:
need_columns = [
    "eligibility",
    "required_documents",
    "interest_and_fees",
    "credit_limit",
    "payment_and_installment",
    "insurance_premium",
    "coverage_and_conditions"
]

customer_needs = (
    conversation_results[need_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("conversation_count")
    .reset_index()
    .rename(columns={"index": "customer_need"})
)

customer_needs["customer_need"] = customer_needs[
    "customer_need"
].map(category_labels)

customer_needs

,customer_need,conversation_count
0,Coverage and conditions,166
1,Payment and installment,145
2,Required documents,122
3,Interest and fees,37
4,Credit limit,36
5,Eligibility,31
6,Insurance premium,23


Direct answer:

In [18]:
top_customer_needs = customer_needs.head(5)[
    "customer_need"
].tolist()

print("The most common detected customer needs were:")
print(", ".join(top_customer_needs))

The most common detected customer needs were:
Coverage and conditions, Payment and installment, Required documents, Interest and fees, Credit limit


Expected outcome 5: Marketing insights

In [19]:
conversation_results["has_any_match"] = conversation_results[
    category_columns
].any(axis=1)

matched_conversations = conversation_results[
    "has_any_match"
].sum()

unmatched_conversations = (
    ~conversation_results["has_any_match"]
).sum()

print("Matched conversations:", matched_conversations)
print("Unmatched conversations:", unmatched_conversations)

Matched conversations: 811
Unmatched conversations: 95


Generate preliminary recommendations:

In [20]:
print("Preliminary marketing insights:\n")

print(
    "1. Prioritize campaign information about:",
    ", ".join(top_customer_needs[:3])
)

print(
    "2. Improve chatbot support for:",
    ", ".join(top_contact_reasons[:3])
)

print(
    "3. Review unmatched conversations to identify",
    "new customer needs and missing keywords."
)

print(
    "4. Clearly explain eligibility, conditions,",
    "fees, and application requirements in campaigns."
)

Preliminary marketing insights:

1. Prioritize campaign information about: Coverage and conditions, Payment and installment, Required documents
2. Improve chatbot support for: Coverage and conditions, Branch or contact, Payment and installment
3. Review unmatched conversations to identify new customer needs and missing keywords.
4. Clearly explain eligibility, conditions, fees, and application requirements in campaigns.
